# Linear head vs cosine head

在同一份 Stage 14 过滤后 crop dataset、同一 DINOv3 冻结特征和同一 train/validation 划分上，快速比较：

- `LinearHead`: 普通 `nn.Linear`，包含 bias；
- `CosineHead`: 对 feature 和类别权重做 L2 normalize，以 cosine similarity 分类，并学习正的 logit scale。

Notebook 使用独立的 `linear_vs_cosine_features.pt`：首次运行从当前 dataset 抽取一次 backbone 特征，后续运行直接复用，且不会覆盖正式 trainer 缓存。两种 head 使用相同初始化 seed、batch 顺序、optimizer 配置、epoch 上限与 macro-F1 early stopping。这个实验比较的是 **head 的训练方式**，不是 nearest-centroid/prototype retrieval。

In [ ]:
from copy import deepcopy
from pathlib import Path
import math
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import AutoImageProcessor


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "pipeline").exists():
            return candidate
    raise RuntimeError(f"Cannot find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_core.model import BackboneLinearClassifier
from pipeline import utils
from trainer.train import (
    get_amp_dtype, get_torch_device, load_or_create_feature_cache, load_tables,
    split_metadata,
)

config = utils.load_pipeline_config()
trainer_cfg = config["trainer"]
device = get_torch_device(trainer_cfg["device"])
print(f"project={PROJECT_ROOT}\ndevice={device}")

In [ ]:
# 快速实验参数；不修改正式 pipeline config。
SEED = int(trainer_cfg["seed"])
CACHE_FILE_NAME = "linear_vs_cosine_features.pt"
REBUILD_FEATURE_CACHE = False
BATCH_SIZE = 256
MAX_EPOCHS = 50
PATIENCE = 6
MIN_DELTA = 1e-4
LEARNING_RATE = float(trainer_cfg["phases"][0]["learning_rate"])
WEIGHT_DECAY = float(trainer_cfg["phases"][0]["weight_decay"])
TOP_K = tuple(int(k) for k in trainer_cfg["top_k"])
USE_CLASS_WEIGHTS = False  # 长尾敏感度实验可改为 True；两种 head 会共享同一权重。
COSINE_TEMPERATURE = 0.07

feature_cache_path = utils.join_data_root(
    trainer_cfg["feature_cache_dir"], config=config
) / CACHE_FILE_NAME
print(feature_cache_path)

## 加载并严格校验特征缓存

缓存必须与当前 backbone、pooling、image size、train/validation split 路径顺序及标签一致。独立缓存不存在时会自动抽取；dataset 或标签改变后，把 `REBUILD_FEATURE_CACHE=True` 再运行本单元。

In [ ]:
dataset_root, metadata, labels_table = load_tables(config)
train_df, val_df = split_metadata(metadata=metadata, trainer_cfg=trainer_cfg)
phase_cfg = dict(trainer_cfg["phases"][0])
phase_cfg["feature_cache_file_name"] = CACHE_FILE_NAME
phase_cfg["feature_cache_rebuild"] = REBUILD_FEATURE_CACHE

if REBUILD_FEATURE_CACHE or not feature_cache_path.exists():
    processor = AutoImageProcessor.from_pretrained(trainer_cfg["backbone_model_name"])
    backbone_model = BackboneLinearClassifier(
        backbone_model_name=trainer_cfg["backbone_model_name"],
        num_classes=len(labels_table), freeze_backbone=True,
    )
    cache = load_or_create_feature_cache(
        config=config, trainer_cfg=trainer_cfg, phase_cfg=phase_cfg, model=backbone_model,
        train_df=train_df, val_df=val_df, dataset_root=dataset_root, processor=processor,
        device=device, amp_enabled=bool(trainer_cfg["amp_enabled"]) and device.type == "cuda",
        amp_dtype=get_amp_dtype(trainer_cfg["amp_dtype"]),
    )
    del backbone_model, processor
    if device.type == "cuda":
        torch.cuda.empty_cache()
else:
    cache = torch.load(feature_cache_path, map_location="cpu", weights_only=False)

expected = {
    "backbone_model_name": trainer_cfg["backbone_model_name"],
    "feature_pooling": "cls_patch_mean",
    "image_size": int(trainer_cfg["image_size"]),
}
for key, value in expected.items():
    if cache.get(key) != value:
        raise ValueError(f"cache {key} mismatch: {cache.get(key)!r} != {value!r}")

path_column = trainer_cfg["image_path_column"]
label_column = trainer_cfg["label_id_column"]
for split, frame in (("train", train_df), ("val", val_df)):
    expected_paths = frame[path_column].astype(str).str.replace("\\", "/", regex=False).tolist()
    expected_labels = torch.tensor(frame[label_column].astype(int).to_numpy())
    if cache[split]["image_paths"] != expected_paths:
        raise ValueError(f"{split} image path/order mismatch; set REBUILD_FEATURE_CACHE=True")
    if not torch.equal(cache[split]["labels"].long(), expected_labels.long()):
        raise ValueError(f"{split} labels mismatch; set REBUILD_FEATURE_CACHE=True")
    if not torch.isfinite(cache[split]["features"]).all():
        raise ValueError(f"{split} contains non-finite features")

num_classes = len(labels_table)
feature_dim = int(cache["feature_dim"])
label_names = labels_table.set_index("label_id")["label"].astype(str).to_dict()
summary = pd.DataFrame({
    "split": ["train", "val"],
    "samples": [len(cache["train"]["labels"]), len(cache["val"]["labels"])],
    "classes_present": [cache[s]["labels"].unique().numel() for s in ("train", "val")],
})
print(f"feature_dim={feature_dim}, num_classes={num_classes}")
display(summary)

In [ ]:
class LinearHead(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.classifier(features)


class CosineHead(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, temperature: float = 0.07):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_classes, input_dim))
        nn.init.normal_(self.weight, std=0.01)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / temperature)))

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        cosine = F.linear(F.normalize(features, dim=-1), F.normalize(self.weight, dim=-1))
        scale = self.logit_scale.clamp(max=math.log(100.0)).exp()
        return scale * cosine


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loader(split: str, *, shuffle: bool) -> DataLoader:
    table = cache[split]
    dataset = TensorDataset(table["features"].float(), table["labels"].long())
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator)


train_counts = torch.bincount(cache["train"]["labels"].long(), minlength=num_classes).float()
class_weights = train_counts.sum() / (num_classes * train_counts.clamp_min(1))
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if USE_CLASS_WEIGHTS else None)

In [ ]:
@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[dict, pd.DataFrame]:
    model.eval()
    losses, truths, predictions, confidences = [], [], [], []
    topk_correct = {k: 0 for k in TOP_K}
    n = 0
    for features_cpu, labels_cpu in loader:
        features, labels = features_cpu.to(device), labels_cpu.to(device)
        logits = model(features)
        losses.append(F.cross_entropy(logits, labels, reduction="sum").item())
        probs = logits.softmax(dim=1)
        confidence, pred = probs.max(dim=1)
        max_k = min(max(TOP_K), num_classes)
        ranked = logits.topk(max_k, dim=1).indices
        for k in TOP_K:
            topk_correct[k] += ranked[:, :min(k, num_classes)].eq(labels[:, None]).any(dim=1).sum().item()
        truths.extend(labels.cpu().tolist())
        predictions.extend(pred.cpu().tolist())
        confidences.extend(confidence.cpu().tolist())
        n += labels.numel()
    pred_df = pd.DataFrame({"label_id": truths, "pred_id": predictions, "confidence": confidences})
    metrics = {
        "loss": sum(losses) / n,
        "accuracy": accuracy_score(truths, predictions),
        "macro_f1": f1_score(truths, predictions, labels=range(num_classes), average="macro", zero_division=0),
        "weighted_f1": f1_score(truths, predictions, labels=range(num_classes), average="weighted", zero_division=0),
        **{f"top_{k}_accuracy": topk_correct[k] / n for k in TOP_K},
    }
    return metrics, pred_df


def train_head(name: str, model: nn.Module) -> dict:
    set_seed(SEED)
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    train_loader = make_loader("train", shuffle=True)
    val_loader = make_loader("val", shuffle=False)
    best_state, best_score, stale = None, -math.inf, 0
    history = []
    for epoch in tqdm(range(1, MAX_EPOCHS + 1), desc=name):
        model.train()
        loss_sum, correct, n = 0.0, 0, 0
        for features_cpu, labels_cpu in train_loader:
            features, labels = features_cpu.to(device), labels_cpu.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * labels.numel()
            correct += logits.argmax(dim=1).eq(labels).sum().item()
            n += labels.numel()
        val_metrics, _ = evaluate(model, val_loader)
        history.append({"head": name, "epoch": epoch, "train_loss": loss_sum / n, "train_accuracy": correct / n, **{f"val_{k}": v for k, v in val_metrics.items()}})
        if val_metrics["macro_f1"] > best_score + MIN_DELTA:
            best_score, stale = val_metrics["macro_f1"], 0
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
        else:
            stale += 1
        if stale >= PATIENCE:
            break
    model.load_state_dict(best_state)
    metrics, predictions = evaluate(model, val_loader)
    return {"name": name, "model": model.cpu(), "best_epoch": best_epoch, "metrics": metrics, "predictions": predictions, "history": pd.DataFrame(history)}

## 训练两种 head

验证集同时用于 early stopping 和比较，因此结果适合快速选方向，不应当作最终无偏 test 指标。若要做正式结论，应固定方案后增加独立 test split 或交叉验证。

In [ ]:
set_seed(SEED)
linear_head = LinearHead(feature_dim, num_classes)
set_seed(SEED)
cosine_head = CosineHead(feature_dim, num_classes, COSINE_TEMPERATURE)
experiments = {
    "linear": train_head("linear", linear_head),
    "cosine": train_head("cosine", cosine_head),
}

metric_columns = ["accuracy", "macro_f1", "weighted_f1", *[f"top_{k}_accuracy" for k in TOP_K]]
comparison = pd.DataFrame([
    {"head": name, "best_epoch": result["best_epoch"], **result["metrics"]}
    for name, result in experiments.items()
]).set_index("head")
display(comparison[["best_epoch", "loss", *metric_columns]].style.format(precision=4))
display((comparison.loc["cosine", metric_columns] - comparison.loc["linear", metric_columns]).rename("cosine_minus_linear").to_frame().style.format(precision=4))

In [ ]:
history = pd.concat([result["history"] for result in experiments.values()], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for head, frame in history.groupby("head"):
    axes[0].plot(frame["epoch"], frame["val_macro_f1"], marker=".", label=head)
    axes[1].plot(frame["epoch"], frame["val_accuracy"], marker=".", label=head)
axes[0].set(title="Validation macro F1", xlabel="epoch")
axes[1].set(title="Validation accuracy", xlabel="epoch")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()

## 每类表现与分歧样本

`f1_delta > 0` 表示 cosine head 在该类更好。分歧表尤其适合检查 cosine 的单位球约束是否改善长尾类，或是否牺牲了容易类别。

In [ ]:
def per_class_frame(result: dict, suffix: str) -> pd.DataFrame:
    pred = result["predictions"]
    report = classification_report(
        pred["label_id"], pred["pred_id"], labels=list(range(num_classes)),
        output_dict=True, zero_division=0,
    )
    rows = [{"label_id": label_id, "label": label_names[label_id], **report[str(label_id)]} for label_id in range(num_classes)]
    return pd.DataFrame(rows).rename(columns={c: f"{c}_{suffix}" for c in ("precision", "recall", "f1-score")})

per_class = per_class_frame(experiments["linear"], "linear").merge(
    per_class_frame(experiments["cosine"], "cosine"), on=["label_id", "label", "support"]
)
per_class["f1_delta"] = per_class["f1-score_cosine"] - per_class["f1-score_linear"]
display(per_class.sort_values("f1_delta", ascending=False).head(20))
display(per_class.sort_values("f1_delta").head(20))

linear_pred = experiments["linear"]["predictions"].add_suffix("_linear")
cosine_pred = experiments["cosine"]["predictions"].add_suffix("_cosine")
disagreements = pd.concat([linear_pred, cosine_pred], axis=1)
disagreements["image_path"] = cache["val"]["image_paths"]
disagreements["true_label"] = disagreements["label_id_linear"].map(label_names)
disagreements["linear_label"] = disagreements["pred_id_linear"].map(label_names)
disagreements["cosine_label"] = disagreements["pred_id_cosine"].map(label_names)
disagreements = disagreements[disagreements["pred_id_linear"] != disagreements["pred_id_cosine"]].copy()
display(disagreements[["image_path", "true_label", "linear_label", "cosine_label", "confidence_linear", "confidence_cosine"]].head(50))